In [1]:
# !pip install tensorflow opencv-python pygame scikit-learn

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import cv2
import pygame
from sklearn.metrics import accuracy_score
import threading
from tensorflow.keras.preprocessing import image

pygame 2.6.1 (SDL 2.28.4, Python 3.10.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
IMG_SIZE = (24, 24)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    'data/train',  # Update path if needed
    target_size=IMG_SIZE,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='training'
)

val_data = datagen.flow_from_directory(
    'data/train',  # Update path if needed
    target_size=IMG_SIZE,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='validation'
)

Found 8923 images belonging to 2 classes.
Found 2229 images belonging to 2 classes.


In [4]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(24,24,1)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='softmax')  # 2 classes: open, closed
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

# Save the model
model.save('models/drowsiness_model.keras')
print("Model trained and saved!")

Epoch 1/20
279/279 [==============================] - 12s 42ms/step - loss: 0.1433 - accuracy: 0.9346 - val_loss: 0.2512 - val_accuracy: 0.9134
Epoch 2/20
279/279 [==============================] - 11s 41ms/step - loss: 0.0332 - accuracy: 0.9889 - val_loss: 0.4216 - val_accuracy: 0.9287
Epoch 3/20
279/279 [==============================] - 12s 42ms/step - loss: 0.0273 - accuracy: 0.9901 - val_loss: 0.4635 - val_accuracy: 0.9273
Epoch 4/20
279/279 [==============================] - 12s 43ms/step - loss: 0.0190 - accuracy: 0.9943 - val_loss: 0.3009 - val_accuracy: 0.9318
Epoch 5/20
279/279 [==============================] - 13s 46ms/step - loss: 0.0163 - accuracy: 0.9952 - val_loss: 0.5132 - val_accuracy: 0.9157
Epoch 6/20
279/279 [==============================] - 12s 43ms/step - loss: 0.0116 - accuracy: 0.9969 - val_loss: 0.3553 - val_accuracy: 0.9300
Epoch 7/20
279/279 [==============================] - 13s 45ms/step - loss: 0.0128 - accuracy: 0.9962 - val_loss: 0.4481 - val_accuracy:

In [5]:
model = tf.keras.models.load_model('models/drowsiness_model.keras')

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    'data/test',  # Update path if needed
    target_size=IMG_SIZE,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

predictions = model.predict(test_data)
predicted_labels = predictions.argmax(axis=1)
true_labels = test_data.classes
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Found 11153 images belonging to 2 classes.
349/349 [==============================] - 8s 21ms/step
Test Accuracy: 98.40%


In [6]:
sample_image_path = 'data/test/open/6a00e2aa-9e4f-11ec-849c-842b2bb3a5b6.jpg'  # Update with real path

img = image.load_img(sample_image_path, target_size=IMG_SIZE, color_mode='grayscale')
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
label = "Open" if prediction.argmax() == 1 else "Closed"
print(f"Prediction for sample image: {label}")

1/1 [==============================] - 0s 80ms/step
Prediction for sample image: Open


In [11]:
# # Initialize Pygame for sound (optional)
# pygame.mixer.init()

# # Load cascades (built-in to OpenCV)
# face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
# eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

# # Function to play alert in a thread
# def play_alert():
#     sound = pygame.mixer.Sound('sounds/alert.mp3')  # Update path if needed
#     sound.play()

# # Start webcam
# cap = cv2.VideoCapture(0)  # 0 for default webcam
# drowsy_counter = 0q

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
#     for (x, y, w, h) in faces:
#         face_roi = gray[y:y+h, x:x+w]
#         eyes = eye_cascade.detectMultiScale(face_roi)
#         for (ex, ey, ew, eh) in eyes:
#             eye_roi = face_roi[ey:ey+eh, ex:ex+ew]
#             eye_roi = cv2.resize(eye_roi, (24, 24)) / 255.0
#             eye_roi = np.expand_dims(eye_roi, axis=(0, -1))  # Add batch and channel dims
#             prediction = model.predict(eye_roi)
#             label = "Open" if prediction.argmax() == 1 else "Closed"
            
#             if label == "Closed":
#                 drowsy_counter += 1
#             else:
#                 drowsy_counter = 0
            
#             if drowsy_counter > 15:
#                 threading.Thread(target=play_alert, daemon=True).start()
#                 cv2.putText(frame, "DROWSINESS ALERT!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    
#     cv2.imshow("Eye Detection", frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()

# Initialize Pygame for sound
import pygame
import cv2
import numpy as np
import time
import threading
from tensorflow.keras.preprocessing import image
import tensorflow as tf

pygame.mixer.init()

# Load cascades (built-in to OpenCV)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

# Load the trained model
model = tf.keras.models.load_model('models/drowsiness_model.keras')

# Function to play alert in a thread
def play_alert():
    sound = pygame.mixer.Sound('sounds/alert.mp3')  # Update path if needed
    pygame.mixer.Sound.play(sound)

# Start webcam
cap = cv2.VideoCapture(0)  # 0 for default webcam
closed_eyes_start_time = None
CLOSED_EYES_THRESHOLD = 5.0  # Seconds to trigger alert
alert_playing = False

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    eyes_closed = False  # Track if any eye is closed in this frame
    
    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(face_roi)
        for (ex, ey, ew, eh) in eyes:
            eye_roi = face_roi[ey:ey+eh, ex:ex+ew]
            eye_roi = cv2.resize(eye_roi, (24, 24)) / 255.0
            eye_roi = np.expand_dims(eye_roi, axis=(0, -1))  # Add batch and channel dims
            prediction = model.predict(eye_roi)
            label = "Opened" if prediction.argmax() == 1 else "Closed"
            
            if label == "Closed":
                eyes_closed = True
            # Display label on frame
            cv2.putText(frame, label, (x + ex, y + ey - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Time-based logic for alert
    if eyes_closed:
        if closed_eyes_start_time is None:
            closed_eyes_start_time = time.time()  # Start timer
        elif time.time() - closed_eyes_start_time >= CLOSED_EYES_THRESHOLD and not alert_playing:
            threading.Thread(target=play_alert, daemon=True).start()
            alert_playing = True
            cv2.putText(frame, "DROWSINESS ALERT!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    else:
        if alert_playing:
            pygame.mixer.stop()  # Stop alert sound when eyes open
            alert_playing = False
        closed_eyes_start_time = None  # Reset timer
    
    cv2.imshow("Eye Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pygame.mixer.quit()

1/1 [==============================] - 0s 27ms/step
